<a href="https://colab.research.google.com/github/sachin886x/deep-learning-lab/blob/main/experiment9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install wandb huggingface_hub -q

import wandb
wandb.login()

from huggingface_hub import notebook_login
notebook_login()

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import os

from torch.utils.data import DataLoader, random_split
from huggingface_hub import HfApi, create_repo

HF_REPO = "mraidtu/exp9-gan-dcgan"
WANDB_PROJECT = "exp9-gan-dcgan"

EPOCHS = 4
BATCH_SIZE = 128
LATENT_DIM = 100
IMAGE_SIZE = 28
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LOSSES = ["bce", "lsgan", "wgan"]
OPTIMIZERS = ["adam", "sgd", "rmsprop"]
MODELS = ["gan", "dcgan"]

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # [-1,1]
])

dataset = torchvision.datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transform
)

train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

class Generator_GAN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(LATENT_DIM, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 784),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z).view(-1, 1, 28, 28)


class Discriminator_GAN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

class Generator_DCGAN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(LATENT_DIM, 128, 7, 1, 0),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.ConvTranspose2d(64, 1, 4, 2, 1),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z.view(-1, LATENT_DIM, 1, 1))


class Discriminator_DCGAN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 64, 4, 2, 1),
            nn.LeakyReLU(0.2),

            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),

            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

def get_loss(name):
    if name == "bce":
        return nn.BCELoss()
    elif name == "lsgan":
        return nn.MSELoss()
    elif name == "wgan":
        return None

def get_optimizer(params, name):
    if name == "adam":
        return optim.Adam(params, lr=0.0002, betas=(0.5, 0.999))
    elif name == "sgd":
        return optim.SGD(params, lr=0.01)
    else:
        return optim.RMSprop(params, lr=0.0002)

def run_experiment(model_type, loss_type, opt_name):
    name = f"{model_type}_{loss_type}_{opt_name}"
    print(f"\nStarting: {name}")

    wandb.init(project=WANDB_PROJECT, name=name)

    if model_type == "gan":
        G = Generator_GAN().to(DEVICE)
        D = Discriminator_GAN().to(DEVICE)
    else:
        G = Generator_DCGAN().to(DEVICE)
        D = Discriminator_DCGAN().to(DEVICE)

    criterion = get_loss(loss_type)

    opt_G = get_optimizer(G.parameters(), opt_name)
    opt_D = get_optimizer(D.parameters(), opt_name)

    for epoch in range(EPOCHS):
        for real, _ in train_loader:
            real = real.to(DEVICE)
            batch = real.size(0)

            # =================================================
            # 🔥 Train Discriminator
            # =================================================
            opt_D.zero_grad()

            z = torch.randn(batch, LATENT_DIM).to(DEVICE)
            fake = G(z)

            if loss_type == "wgan":
                loss_D = -(torch.mean(D(real)) - torch.mean(D(fake)))
            else:
                real_labels = torch.ones(batch, 1).to(DEVICE)
                fake_labels = torch.zeros(batch, 1).to(DEVICE)

                loss_real = criterion(D(real), real_labels)
                loss_fake = criterion(D(fake.detach()), fake_labels)
                loss_D = loss_real + loss_fake

            loss_D.backward()
            opt_D.step()

            # =================================================
            # 🔥 Train Generator (FIXED)
            # =================================================
            opt_G.zero_grad()

            # IMPORTANT FIX 🔥 (new fake sample)
            z = torch.randn(batch, LATENT_DIM).to(DEVICE)
            fake = G(z)

            if loss_type == "wgan":
                loss_G = -torch.mean(D(fake))
            else:
                real_labels = torch.ones(batch, 1).to(DEVICE)
                loss_G = criterion(D(fake), real_labels)

            loss_G.backward()
            opt_G.step()

        wandb.log({
            "epoch": epoch,
            "G_loss": loss_G.item(),
            "D_loss": loss_D.item()
        })

        print(f"Epoch {epoch}: G={loss_G:.4f} D={loss_D:.4f}")

    torch.save(G.state_dict(), f"{name}_G.pt")
    wandb.finish()

    return G

results = {}
saved_models = {}

for model in MODELS:
    for loss in LOSSES:
        for opt in OPTIMIZERS:
            G = run_experiment(model, loss, opt)
            saved_models[f"{model}_{loss}_{opt}"] = G

create_repo(HF_REPO, exist_ok=True)
api = HfApi()

for name in saved_models:
    fname = f"{name}_G.pt"
    if os.path.exists(fname):
        api.upload_file(
            path_or_fileobj=fname,
            path_in_repo=f"models/{fname}",
            repo_id=HF_REPO
        )
        print(f"uploaded: {fname}")

username = wandb.api.viewer()['entity']
print(f"W&B : https://wandb.ai/{username}/{WANDB_PROJECT}")
print(f"HF : https://huggingface.co/{HF_REPO}")